# Chapter 9.7 - Backpropagation Through Time

Backpropagation through time applies the chain rule to an unrolled recurrent computation. This notebook makes gradient paths visible, demonstrates vanishing and exploding products, and shows exactly what state detachment truncates.

## How to use this notebook

Run the notebook from top to bottom in a clean kernel. Everything is generated from small tensors or inline text, so there are no downloads. Before important cells, predict the time, batch, feature, vocabulary, and hidden-state shapes. Treat every assertion as an executable contract rather than decoration.

## You are done when you can

- draw an RNN as repeated uses of shared parameters
- explain direct and recurrent gradient paths
- verify a recurrent gradient with autograd
- predict when repeated Jacobian factors vanish or explode
- distinguish gradient clipping from state detachment


In [ ]:
import math
import random
from collections import Counter

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(0)
random.seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)


## 9.7.0 The Problem This Notebook Solves

The forward recurrence reuses the same parameters:

```text
h0 --[w, x1]--> h1 --[w, x2]--> h2 --[w, x3]--> h3
```

**Unrolling** draws each time step as a separate computation even though all steps share `w`. **Backpropagation through time (BPTT)** applies ordinary reverse-mode automatic differentiation to this unrolled graph.

A parameter such as `w` affects the loss through many paths: directly at the current step and indirectly through every later hidden state. The product of many derivatives can become tiny (vanishing gradient) or huge (exploding gradient).


## 9.7.1 A Scalar Recurrence You Can Differentiate

To isolate time effects, use the linear recurrence `h_t = w * h_(t-1)` with `h_0 = 1`. After `T` steps, `h_T = w^T`, so

```text
d h_T / d w = T * w^(T - 1)
```

The factor `T` appears because the shared parameter participates at every step. Autograd should agree with the analytic derivative.


In [ ]:
def final_linear_state(weight, steps):
    state = torch.tensor(1.0)
    for _ in range(steps):
        state = weight * state
    return state

w = torch.tensor(1.2, requires_grad=True)
steps = 5
final_state = final_linear_state(w, steps)
final_state.backward()
analytic = steps * (w.detach() ** (steps - 1))

print("final state:", final_state.item())
print("autograd:", w.grad.item(), "analytic:", analytic.item())
assert torch.allclose(w.grad, analytic)


## 9.7.2 Vanishing and Exploding Through Repeated Products

The sensitivity of a late state to an early state contains repeated local derivatives. In the linear example, `d h_T / d h_0 = w^T`.

- If `|w| < 1`, the product shrinks exponentially.
- If `|w| > 1`, the product grows exponentially.
- Nonlinear RNNs replace the scalar factor with Jacobian matrices, but repeated multiplication creates the same core risk.


In [ ]:
steps = torch.arange(1, 21)
vanishing = 0.5 ** steps
stable = 1.0 ** steps
exploding = 1.5 ** steps

print("at step 20:", {
    "0.5^T": vanishing[-1].item(),
    "1.0^T": stable[-1].item(),
    "1.5^T": exploding[-1].item(),
})
assert vanishing[-1] < 1e-5
assert stable[-1] == 1
assert exploding[-1] > 1000


`tanh` also contributes derivatives. Its derivative is `1 - tanh(z)^2`, at most 1 and near 0 when `z` has large magnitude. Saturated hidden units can therefore weaken long-range gradients even if recurrent weights alone are not small.


In [ ]:
z = torch.tensor([-5.0, -1.0, 0.0, 1.0, 5.0])
tanh_derivative = 1 - torch.tanh(z) ** 2
print(tanh_derivative)
assert torch.allclose(tanh_derivative[2], torch.tensor(1.0))
assert tanh_derivative[0] < 0.001
assert tanh_derivative[-1] < 0.001


## 9.7.3 Full BPTT Versus Truncated BPTT

Detaching a hidden state preserves its numeric value but removes its connection to earlier operations. The first computation below allows the final loss to send gradient through all four recurrent steps. The second detaches after two steps, so the final loss cannot update the earlier part of the graph.

Truncation reduces memory and limits gradient path length. It is an approximation: long-range credit assignment across the cut is removed.


In [ ]:
def two_segment_gradient(detach_between):
    weight = torch.tensor(1.2, requires_grad=True)
    state = torch.tensor(1.0)
    for _ in range(2):
        state = weight * state
    if detach_between:
        state = state.detach()
    for _ in range(2):
        state = weight * state
    state.backward()
    return state.detach(), weight.grad.detach()

full_value, full_gradient = two_segment_gradient(detach_between=False)
truncated_value, truncated_gradient = two_segment_gradient(detach_between=True)
print({"full": full_gradient.item(), "truncated": truncated_gradient.item()})
assert torch.allclose(full_value, truncated_value)
assert full_gradient > truncated_gradient
assert torch.allclose(full_gradient, torch.tensor(4 * 1.2 ** 3))
assert torch.allclose(truncated_gradient, torch.tensor(2 * 1.2 ** 3))


## 9.7.4 Shared Parameters Accumulate Gradient Contributions

When one parameter is used several times, autograd adds the gradient contribution from every path into the same `.grad` tensor. It does not create a separate parameter per time step. Calling `zero_grad()` is necessary before a new optimization step because PyTorch also accumulates gradients across separate backward calls.


In [ ]:
weight = torch.tensor(2.0, requires_grad=True)
state = torch.tensor(1.0)
states = []
for _ in range(3):
    state = weight * state
    states.append(state)
loss = sum(states)
loss.backward()

expected = 1 + 2 * 2.0 + 3 * (2.0 ** 2)
print("gradient from all loss paths:", weight.grad.item())
assert torch.allclose(weight.grad, torch.tensor(expected))


## 9.7.5 Break It Deliberately: Exploding Gradient

This is a controlled failure demonstration. A recurrent multiplier of 2 over 30 steps creates a derivative larger than a billion. Gradient clipping can cap the stored gradient before an optimizer step, but it does not change the unstable forward dynamics or restore information lost to vanishing gradients.


In [ ]:
weight = torch.tensor(2.0, requires_grad=True)
state = final_linear_state(weight, steps=30)
state.backward()
raw_gradient = weight.grad.item()

torch.nn.utils.clip_grad_norm_([weight], max_norm=1.0)
clipped_gradient = weight.grad.item()
print({"raw": raw_gradient, "clipped": clipped_gradient})
assert raw_gradient > 1e9
assert abs(clipped_gradient) <= 1.00001


## 9.7.6 Practical Interpretation

Three mechanisms are easy to confuse:

- **BPTT** computes gradients through the unrolled recurrence.
- **State detachment** chooses where that graph is cut, limiting history and memory use.
- **Gradient clipping** rescales gradients after backward computation, before the optimizer step.

None is a substitute for the others. Modern gated recurrent cells in Chapter 10 add learnable paths designed to preserve useful information more effectively, but they still use backpropagation through time.


## 9.7 Checkpoint

Answer these without rerunning the notebook. Short markdown answers are enough.

1. Why does unrolling not create new parameters at every time step?
2. Where does the factor T come from in d(w^T)/dw?
3. Why can repeated Jacobian multiplication harm long-range learning?
4. What changes and what stays identical when state is detached?
5. Why can clipping contain an update without fixing the recurrent dynamics?
